# Lab 6: K-means Clustering

This notebook is designed to be reusable.

- Default dataset loads from `data/kmeans_blobs.csv`.
- To use your own dataset, set `DATA_PATH` to a CSV with columns `x1` and `x2`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from sklearn.datasets import make_blobs

# ---- Config (edit these) ----
DATA_PATH = Path("data/kmeans_blobs.csv")
K = 3
MAX_ITERS = 10
INIT_SEED = 42

# Used only when DATA_PATH doesn't exist
GEN_N_SAMPLES = 25
GEN_CENTERS = 3
GEN_CLUSTER_STD = 1.0
GEN_RANDOM_STATE = 42

# ---- Load or generate data ----
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    if not {"x1", "x2"}.issubset(df.columns):
        raise ValueError(f"{DATA_PATH} must have columns: x1, x2")
    X = df[["x1", "x2"]].to_numpy(dtype=float)
else:
    X, _ = make_blobs(
        n_samples=GEN_N_SAMPLES,
        centers=GEN_CENTERS,
        cluster_std=GEN_CLUSTER_STD,
        random_state=GEN_RANDOM_STATE,
    )
    print(f"Note: {DATA_PATH} not found; using generated blobs instead.")

# Visualize the initial dataset
plt.scatter(X[:, 0], X[:, 1], s=50)
plt.title("Initial Data")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()


def kmeans_algorithm(X, k=3, max_iters=10, init_seed=42):
    rng = np.random.default_rng(init_seed)

    # Initialize centroids
    centroids = X[rng.choice(X.shape[0], size=k, replace=False)]

    for iteration in range(max_iters):
        # Assign each data point to the nearest centroid
        distances = np.linalg.norm(X[:, np.newaxis] - centroids, axis=2)
        labels = np.argmin(distances, axis=1)

        # Update centroids
        new_centroids = np.array([X[labels == i].mean(axis=0) for i in range(k)])

        # Plot current clustering
        plt.scatter(X[:, 0], X[:, 1], c=labels, s=50, cmap="viridis")
        plt.scatter(new_centroids[:, 0], new_centroids[:, 1], c="red", s=200, marker="X")
        plt.title(f"K-means Clustering (Iteration {iteration + 1})")
        plt.xlabel("Feature 1")
        plt.ylabel("Feature 2")
        plt.show()

        # Check convergence
        if np.allclose(centroids, new_centroids):
            print(f"Centroids converged at iteration {iteration + 1}")
            break

        centroids = new_centroids


kmeans_algorithm(X, k=K, max_iters=MAX_ITERS, init_seed=INIT_SEED)
